# Model Hyperparameter Tuning

note: model is only tuned with actual labels, the soft labels are not used

| Model / Ensemble | Mean ROC-AUC | Median ROC-AUC | Std ROC-AUC | Worst ROC-AUC | Latest 5 ROC-AUC |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **lgbm_dominant** | **0.995475** | 0.997577 | **0.008006** | **0.968162** | **0.997855** |
| **LightGBM** | 0.995398 | **0.997751** | 0.008123 | 0.967539 | 0.997679 |
| **boosting_dominant** | 0.995395 | 0.997441 | 0.008160 | 0.967572 | 0.997854 |
| **equal** | 0.995200 | 0.997398 | 0.008124 | 0.967613 | 0.997597 |
| **extra_trees_diverse** | 0.995151 | 0.997459 | 0.008023 | 0.967954 | 0.997418 |
| **XGBoost** | 0.994491 | 0.996804 | 0.009932 | 0.960417 | 0.997647 |
| **ExtraTrees** | 0.990361 | 0.994626 | 0.010946 | 0.957531 | 0.991049 |

Result: 
* the ensemble only marginally improved LightGBM's standalone performance. Thus I will be keeping the simpler LightGBM Model
* Best LightGBM hyperparameters: `{'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.1, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05}`
  * there is L2 regularisation (`lambda`), but no L1 rgularisation (best combination was `0.0`)
* Best ExtraTrees hyperparameters: `{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}`
* Best XGBoost hyperparameter: `{'model__subsample': 1.0, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 300, 'model__min_child_weight': 5, 'model__max_depth': 7, 'model__learning_rate': 0.03, 'model__gamma': 0.0, 'model__colsample_bytree': 0.8}`
  * there is L2 regularisation (`lambda`)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline

from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier


In [2]:
# Update these paths if your files are stored elsewhere.
train_raw = pd.read_csv("train.csv")
train_soft_label_raw = pd.read_csv("train_soft_label.csv")
test_raw = pd.read_csv("test.csv")
edges_raw = pd.read_csv("txs_edgelist.csv")

# Confirmed rows are used for temporal validation.
labelled_train = train_soft_label_raw[
    train_soft_label_raw["label"].notna()
].copy()

print("Confirmed labelled rows:", len(labelled_train))
print(
    "Confirmed time range:",
    labelled_train["time_step"].min(),
    "to",
    labelled_train["time_step"].max(),
)

pd.set_option("display.max_colwidth", None)

Confirmed labelled rows: 31235
Confirmed time range: 1 to 35


In [3]:
# Temporal graph setup.
node_times = pd.concat(
    [
        train_raw[["txId", "time_step"]],
        test_raw[["txId", "time_step"]],
    ],
    ignore_index=True,
).drop_duplicates("txId").set_index("txId")["time_step"]

edges = edges_raw[["txId1", "txId2"]].copy()
edges["time_1"] = edges["txId1"].map(node_times)
edges["time_2"] = edges["txId2"].map(node_times)

graph_feature_cols = [
    "in_degree",
    "out_degree",
    "total_degree",
    "degree_ratio",
]

def temporal_node_features(cutoff_step):
    usable_edges = edges[
        edges["time_1"].notna()
        & edges["time_2"].notna()
        & (edges["time_1"] <= cutoff_step)
        & (edges["time_2"] <= cutoff_step)
    ]

    in_degree = usable_edges.groupby("txId2").size().rename("in_degree")
    out_degree = usable_edges.groupby("txId1").size().rename("out_degree")

    features = pd.concat([in_degree, out_degree], axis=1).fillna(0)
    features["total_degree"] = (
        features["in_degree"] + features["out_degree"]
    )
    features["degree_ratio"] = (
        features["out_degree"]
        / features["in_degree"].replace(0, np.nan)
    )
    features["degree_ratio"] = (
        features["degree_ratio"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features.reset_index().rename(columns={"index": "txId"})

def add_temporal_graph_features(frame, cutoff_step):
    graph_features = temporal_node_features(cutoff_step)
    result = frame.merge(graph_features, on="txId", how="left")
    result[graph_feature_cols] = result[graph_feature_cols].fillna(0)
    return result

def calculate_recency_weights(time_steps, reference_time_step, decay_rate):
    age = reference_time_step - time_steps
    return np.exp(-decay_rate * age)


In [4]:
feature_cols = [
    c for c in train_raw.columns
    if c not in ["index", "txId", "label", "time_step"]
]

USE_GRAPH_FEATURES = True
USE_TIME_STEP = False
INITIAL_TRAIN_END = 20
FINAL_VALIDATION_STEP = 35
RECENCY_DECAY = 0.03

print("Ordinary feature count:", len(feature_cols))


Ordinary feature count: 165


In [5]:
def prepare_temporal_folds(
    data,
    feature_cols,
    initial_train_end=20,
    final_validation_step=35,
    recency_decay=0.03,
    use_graph_features=True,
    use_time_step=False,
):
    data = data.copy()
    fold_cache = []

    for validation_step in range(
        initial_train_end + 1,
        final_validation_step + 1,
    ):
        train_end_step = validation_step - 1

        train_fold = data[
            (data["time_step"] <= train_end_step)
            & data["label"].notna()
        ].copy()

        validation_fold = data[
            (data["time_step"] == validation_step)
            & data["label"].notna()
        ].copy()

        if train_fold.empty or validation_fold.empty:
            continue

        y_train = train_fold["label"].astype(int)
        y_valid = validation_fold["label"].astype(int)

        if y_train.nunique() < 2 or y_valid.nunique() < 2:
            continue

        if use_graph_features:
            train_fold = add_temporal_graph_features(
                train_fold,
                cutoff_step=train_end_step,
            )
            validation_fold = add_temporal_graph_features(
                validation_fold,
                cutoff_step=validation_step,
            )

        fold_feature_cols = list(feature_cols)
        if use_graph_features:
            fold_feature_cols += graph_feature_cols
        if use_time_step:
            fold_feature_cols += ["time_step"]

        class_weights = compute_sample_weight( # to control class imbalance
            class_weight="balanced",
            y=y_train,
        )
        recency_weights = calculate_recency_weights(
            train_fold["time_step"].to_numpy(),
            reference_time_step=train_end_step,
            decay_rate=recency_decay,
        )
        sample_weights = class_weights * recency_weights
        sample_weights = sample_weights / sample_weights.mean()

        fold_cache.append({
            "validation_step": validation_step,
            "X_train": train_fold[fold_feature_cols],
            "y_train": y_train,
            "X_valid": validation_fold[fold_feature_cols],
            "y_valid": y_valid,
            "sample_weights": sample_weights,
            "tx_ids": validation_fold["txId"].to_numpy(),
        })

    return fold_cache

fold_cache = prepare_temporal_folds(
    labelled_train,
    feature_cols,
    initial_train_end=INITIAL_TRAIN_END,
    final_validation_step=FINAL_VALIDATION_STEP,
    recency_decay=RECENCY_DECAY,
    use_graph_features=USE_GRAPH_FEATURES,
    use_time_step=USE_TIME_STEP,
)

print("Temporal folds prepared:", len(fold_cache))


Temporal folds prepared: 15


In [6]:
# Hyperparameter spaces for the final tree-based models.
model_specs = {
    "LightGBM": {
        "n_iter": 100,
        "params": {
            "model__n_estimators": [100, 300, 500],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__num_leaves": [15, 31, 63],
            "model__max_depth": [-1, 10, 20],
            "model__min_child_samples": [20, 50, 100],

            # LightGBM regularisation
            "model__reg_alpha": [0.0, 0.1, 1.0],       # L1
            "model__reg_lambda": [0.0, 1.0, 5.0],      # L2
            "model__min_split_gain": [0.0, 0.05, 0.1],
        },
    },
    "XGBoost": {
        "n_iter": 100,
        "params": {
            "model__n_estimators": [100, 300, 500],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_depth": [3, 5, 7],
            "model__min_child_weight": [1, 5, 10],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],

            # XGBoost regularisation
            "model__reg_alpha": [0.0, 0.1, 1.0],       # L1
            "model__reg_lambda": [1.0, 5.0, 10.0],     # L2
            "model__gamma": [0.0, 0.1, 0.5],
        },
    },
    "ExtraTrees": {
        "n_iter": 100,
        "params": {
            "model__n_estimators": [300, 500, 800],
            "model__max_depth": [10, 20, 40],
            "model__min_samples_leaf": [2, 5, 10],
            "model__min_samples_split": [5, 10, 20],
            "model__max_features": ["sqrt", 0.5, 0.8],
            "model__min_impurity_decrease": [0.0, 1e-5, 1e-4],
            "model__class_weight": ["balanced"],
        },
    },
}


In [7]:
def remove_model_prefix(params):
    return {
        key.replace("model__", "", 1): value
        for key, value in params.items()
        if key.startswith("model__")
    }

In [8]:
def create_model(model_name, params):
    model_params = remove_model_prefix(params)

    if model_name == "LightGBM":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                objective="binary",
                random_state=100,
                n_jobs=-1,
                verbosity=-1,
                **model_params,
            )),
        ])

    if model_name == "XGBoost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist", # for fast training speed for massive datasets
                random_state=100,
                n_jobs=-1,
                **model_params,
            )),
        ])

    if model_name == "ExtraTrees":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                random_state=100,
                n_jobs=-1,
                **model_params,
            )),
        ])

    raise ValueError(f"Unknown model: {model_name}")


def fit_model(model_name, model, fold):
    return model.fit(
        fold["X_train"],
        fold["y_train"],
        model__sample_weight=fold["sample_weights"],
    )


def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


In [9]:
def evaluate_configuration(
    model_name,
    params,
    fold_cache,
):
    fold_results = []

    for fold in fold_cache:
        # Create the pipeline with the fixed preprocessor.
        model = create_model(model_name, params)

        model = fit_model(
            model_name=model_name,
            model=model,
            fold=fold,
        )

        scores = get_scores(
            model=model,
            X=fold["X_valid"],
        )

        fold_results.append({
            "validation_step": fold["validation_step"],
            "roc_auc": roc_auc_score(
                fold["y_valid"],
                scores,
            ),
        })

    fold_results = pd.DataFrame(fold_results)

    if fold_results.empty:
        raise ValueError(
            f"No valid validation results were produced "
            f"for {model_name}."
        )

    return {
        "fold_results": fold_results,
        "mean_roc_auc": fold_results["roc_auc"].mean(),
        "median_roc_auc": fold_results["roc_auc"].median(),
        "std_roc_auc": fold_results["roc_auc"].std(),
        "worst_roc_auc": fold_results["roc_auc"].min(),
        "latest_5_roc_auc": (
            fold_results
            .tail(5)["roc_auc"]
            .mean()
        ),
    }


all_model_results = []
best_model_params = {}


for model_name, spec in model_specs.items():

    # Generate every possible parameter combination.
    all_params = list(
    ParameterSampler(
        param_distributions=spec["params"],
        n_iter=spec["n_iter"],
        random_state=100,
    )
)

    print(
        f"{model_name}: "
        f"{len(all_params)} configurations"
    )

    model_results = []

    for trial_number, params in enumerate(
        all_params,
        start=1,
    ):
        print(
            f"{model_name}: "
            f"configuration "
            f"{trial_number}/{len(all_params)}"
        )

        result = evaluate_configuration(
            model_name=model_name,
            params=params,
            fold_cache=fold_cache,
        )

        row = {
            "model": model_name,
            "trial": trial_number,
            "params": params,
            "mean_roc_auc": result["mean_roc_auc"],
            "median_roc_auc": result["median_roc_auc"],
            "std_roc_auc": result["std_roc_auc"],
            "worst_roc_auc": result["worst_roc_auc"],
            "latest_5_roc_auc": result["latest_5_roc_auc"],
        }

        model_results.append(row)
        all_model_results.append(row)

    model_results_df = (
        pd.DataFrame(model_results)
        .sort_values(
            by=[
                "mean_roc_auc",
                "latest_5_roc_auc",
                "worst_roc_auc",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    best_model_params[model_name] = (
        model_results_df.iloc[0]["params"]
    )

    print(f"\nBest {model_name} configuration:")
    display(model_results_df.head(5))

LightGBM: 100 configurations
LightGBM: configuration 1/100
LightGBM: configuration 2/100
LightGBM: configuration 3/100
LightGBM: configuration 4/100
LightGBM: configuration 5/100
LightGBM: configuration 6/100
LightGBM: configuration 7/100
LightGBM: configuration 8/100
LightGBM: configuration 9/100
LightGBM: configuration 10/100
LightGBM: configuration 11/100
LightGBM: configuration 12/100
LightGBM: configuration 13/100
LightGBM: configuration 14/100
LightGBM: configuration 15/100
LightGBM: configuration 16/100
LightGBM: configuration 17/100
LightGBM: configuration 18/100
LightGBM: configuration 19/100
LightGBM: configuration 20/100
LightGBM: configuration 21/100
LightGBM: configuration 22/100
LightGBM: configuration 23/100
LightGBM: configuration 24/100
LightGBM: configuration 25/100
LightGBM: configuration 26/100
LightGBM: configuration 27/100
LightGBM: configuration 28/100
LightGBM: configuration 29/100
LightGBM: configuration 30/100
LightGBM: configuration 31/100
LightGBM: configura

,model,trial,params,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
0,LightGBM,25,"{'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.1, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995398,0.997751,0.008123,0.967539,0.997679
1,LightGBM,32,"{'model__reg_lambda': 0.0, 'model__reg_alpha': 0.1, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.05, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995392,0.997626,0.008469,0.966257,0.997641
2,LightGBM,1,"{'model__reg_lambda': 0.0, 'model__reg_alpha': 0.1, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.1, 'model__min_child_samples': 50, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995242,0.997610,0.008368,0.966640,0.997722
3,LightGBM,72,"{'model__reg_lambda': 0.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 300, 'model__min_split_gain': 0.0, 'model__min_child_samples': 50, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995193,0.997813,0.008730,0.965642,0.998074
4,LightGBM,66,"{'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 300, 'model__min_split_gain': 0.0, 'model__min_child_samples': 50, 'model__max_depth': -1, 'model__learning_rate': 0.05}",0.995191,0.997876,0.008636,0.965725,0.998110


XGBoost: 100 configurations
XGBoost: configuration 1/100
XGBoost: configuration 2/100
XGBoost: configuration 3/100
XGBoost: configuration 4/100
XGBoost: configuration 5/100
XGBoost: configuration 6/100
XGBoost: configuration 7/100
XGBoost: configuration 8/100
XGBoost: configuration 9/100
XGBoost: configuration 10/100
XGBoost: configuration 11/100
XGBoost: configuration 12/100
XGBoost: configuration 13/100
XGBoost: configuration 14/100
XGBoost: configuration 15/100
XGBoost: configuration 16/100
XGBoost: configuration 17/100
XGBoost: configuration 18/100
XGBoost: configuration 19/100
XGBoost: configuration 20/100
XGBoost: configuration 21/100
XGBoost: configuration 22/100
XGBoost: configuration 23/100
XGBoost: configuration 24/100
XGBoost: configuration 25/100
XGBoost: configuration 26/100
XGBoost: configuration 27/100
XGBoost: configuration 28/100
XGBoost: configuration 29/100
XGBoost: configuration 30/100
XGBoost: configuration 31/100
XGBoost: configuration 32/100
XGBoost: configuratio

,model,trial,params,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
0,XGBoost,52,"{'model__subsample': 1.0, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 300, 'model__min_child_weight': 5, 'model__max_depth': 7, 'model__learning_rate': 0.03, 'model__gamma': 0.0, 'model__colsample_bytree': 0.8}",0.994491,0.996804,0.009932,0.960417,0.997647
1,XGBoost,33,"{'model__subsample': 1.0, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 5, 'model__learning_rate': 0.03, 'model__gamma': 0.5, 'model__colsample_bytree': 0.8}",0.994443,0.997604,0.011180,0.955259,0.997415
2,XGBoost,62,"{'model__subsample': 1.0, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.1, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 7, 'model__learning_rate': 0.03, 'model__gamma': 0.0, 'model__colsample_bytree': 0.8}",0.994223,0.996967,0.010541,0.958005,0.997664
3,XGBoost,17,"{'model__subsample': 1.0, 'model__reg_lambda': 10.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 500, 'model__min_child_weight': 5, 'model__max_depth': 7, 'model__learning_rate': 0.1, 'model__gamma': 0.5, 'model__colsample_bytree': 0.8}",0.994203,0.997104,0.012147,0.951250,0.997695
4,XGBoost,72,"{'model__subsample': 1.0, 'model__reg_lambda': 5.0, 'model__reg_alpha': 0.1, 'model__n_estimators': 300, 'model__min_child_weight': 10, 'model__max_depth': 7, 'model__learning_rate': 0.05, 'model__gamma': 0.0, 'model__colsample_bytree': 1.0}",0.994200,0.996745,0.010853,0.956258,0.997328


ExtraTrees: 100 configurations
ExtraTrees: configuration 1/100
ExtraTrees: configuration 2/100
ExtraTrees: configuration 3/100
ExtraTrees: configuration 4/100
ExtraTrees: configuration 5/100
ExtraTrees: configuration 6/100
ExtraTrees: configuration 7/100
ExtraTrees: configuration 8/100
ExtraTrees: configuration 9/100
ExtraTrees: configuration 10/100
ExtraTrees: configuration 11/100
ExtraTrees: configuration 12/100
ExtraTrees: configuration 13/100
ExtraTrees: configuration 14/100
ExtraTrees: configuration 15/100
ExtraTrees: configuration 16/100
ExtraTrees: configuration 17/100
ExtraTrees: configuration 18/100
ExtraTrees: configuration 19/100
ExtraTrees: configuration 20/100
ExtraTrees: configuration 21/100
ExtraTrees: configuration 22/100
ExtraTrees: configuration 23/100
ExtraTrees: configuration 24/100
ExtraTrees: configuration 25/100
ExtraTrees: configuration 26/100
ExtraTrees: configuration 27/100
ExtraTrees: configuration 28/100
ExtraTrees: configuration 29/100
ExtraTrees: configura

,model,trial,params,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
0,ExtraTrees,73,"{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.990361,0.994626,0.010946,0.957531,0.991049
1,ExtraTrees,31,"{'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.989287,0.992885,0.013466,0.945959,0.991158
2,ExtraTrees,38,"{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 'sqrt', 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.988914,0.995568,0.017700,0.928530,0.993359
3,ExtraTrees,16,"{'model__n_estimators': 500, 'model__min_samples_split': 20, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.988801,0.993291,0.013942,0.945559,0.989989
4,ExtraTrees,41,"{'model__n_estimators': 800, 'model__min_samples_split': 20, 'model__min_samples_leaf': 5, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.8, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.988184,0.994130,0.015241,0.943030,0.989231


In [10]:
all_model_results_df = pd.DataFrame(
    all_model_results
)

best_model_summary = (
    all_model_results_df
    .sort_values(
        [
            "model",
            "mean_roc_auc",
            "latest_5_roc_auc",
            "worst_roc_auc",
        ],
        ascending=[
            True,
            False,
            False,
            False,
        ],
    )
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

display(best_model_summary)

,model,trial,params,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
0,ExtraTrees,73,"{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.990361,0.994626,0.010946,0.957531,0.991049
1,LightGBM,25,"{'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.1, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995398,0.997751,0.008123,0.967539,0.997679
2,XGBoost,52,"{'model__subsample': 1.0, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__n_estimators': 300, 'model__min_child_weight': 5, 'model__max_depth': 7, 'model__learning_rate': 0.03, 'model__gamma': 0.0, 'model__colsample_bytree': 0.8}",0.994491,0.996804,0.009932,0.960417,0.997647


In [11]:
top5_per_model = (
    all_model_results_df
    .sort_values(
        by=[
            "model",
            "mean_roc_auc",
            "latest_5_roc_auc",
            "worst_roc_auc",
        ],
        ascending=[
            True,
            False,
            False,
            False,
        ],
    )
    .groupby("model", group_keys=False)
    .head(5)
    .copy()
)

top5_per_model["within_model_rank"] = (
    top5_per_model
    .groupby("model")
    .cumcount()
    + 1
)

top5_per_model = top5_per_model[
    [
        "model",
        "within_model_rank",
        "params",
        "mean_roc_auc",
        "median_roc_auc",
        "std_roc_auc",
        "worst_roc_auc",
        "latest_5_roc_auc",
    ]
]

display(top5_per_model)

,model,within_model_rank,params,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
272,ExtraTrees,1,"{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.990361,0.994626,0.010946,0.957531,0.991049
230,ExtraTrees,2,"{'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.989287,0.992885,0.013466,0.945959,0.991158
237,ExtraTrees,3,"{'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 'sqrt', 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.988914,0.995568,0.017700,0.928530,0.993359
215,ExtraTrees,4,"{'model__n_estimators': 500, 'model__min_samples_split': 20, 'model__min_samples_leaf': 2, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.5, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.988801,0.993291,0.013942,0.945559,0.989989
240,ExtraTrees,5,"{'model__n_estimators': 800, 'model__min_samples_split': 20, 'model__min_samples_leaf': 5, 'model__min_impurity_decrease': 0.0, 'model__max_features': 0.8, 'model__max_depth': 40, 'model__class_weight': 'balanced'}",0.988184,0.994130,0.015241,0.943030,0.989231
24,LightGBM,1,"{'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.1, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995398,0.997751,0.008123,0.967539,0.997679
31,LightGBM,2,"{'model__reg_lambda': 0.0, 'model__reg_alpha': 0.1, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.05, 'model__min_child_samples': 20, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995392,0.997626,0.008469,0.966257,0.997641
0,LightGBM,3,"{'model__reg_lambda': 0.0, 'model__reg_alpha': 0.1, 'model__num_leaves': 15, 'model__n_estimators': 500, 'model__min_split_gain': 0.1, 'model__min_child_samples': 50, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995242,0.997610,0.008368,0.966640,0.997722
71,LightGBM,4,"{'model__reg_lambda': 0.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 300, 'model__min_split_gain': 0.0, 'model__min_child_samples': 50, 'model__max_depth': 10, 'model__learning_rate': 0.05}",0.995193,0.997813,0.008730,0.965642,0.998074
65,LightGBM,5,"{'model__reg_lambda': 1.0, 'model__reg_alpha': 0.0, 'model__num_leaves': 15, 'model__n_estimators': 300, 'model__min_split_gain': 0.0, 'model__min_child_samples': 50, 'model__max_depth': -1, 'model__learning_rate': 0.05}",0.995191,0.997876,0.008636,0.965725,0.998110


In [12]:
# Generate out-of-fold scores for the best configuration of each model.
oof_scores = []

for fold in fold_cache:
    fold_output = pd.DataFrame({
        "txId": fold["tx_ids"],
        "validation_step": fold["validation_step"],
        "y_true": fold["y_valid"].to_numpy(),
    })

    for model_name, params in best_model_params.items():
        model = create_model(model_name, params)
        model = fit_model(model_name, model, fold)
        fold_output[model_name] = get_scores(
            model,
            fold["X_valid"],
        )

    oof_scores.append(fold_output)

oof_scores = pd.concat(oof_scores, ignore_index=True)

for model_name in best_model_params:
    oof_scores[f"{model_name}_rank"] = (
        oof_scores
        .groupby("validation_step")[model_name]
        .rank(pct=True)
    )

display(oof_scores.head())


,txId,validation_step,y_true,LightGBM,XGBoost,ExtraTrees,LightGBM_rank,XGBoost_rank,ExtraTrees_rank
0,98975362,21,0,0.005737,0.004031,0.015874,0.689548,0.407176,0.566303
1,100074684,21,1,0.986751,0.895305,0.215624,0.865835,0.859594,0.808112
2,114011110,21,0,0.000106,0.001493,0.000000,0.210608,0.184087,0.109984
3,99276501,21,0,0.013118,0.045417,0.115478,0.734789,0.717629,0.764431
4,100084315,21,0,0.000110,0.002041,0.000000,0.218409,0.243370,0.109984


In [20]:
ensemble_weight_options = {
    "equal": {
        "LightGBM_rank": 1/3,
        "XGBoost_rank": 1/3,
        "ExtraTrees_rank": 1/3,
    },
    "lgbm_dominant": {
        "LightGBM_rank": 0.5,
        "XGBoost_rank": 0.3,
        "ExtraTrees_rank": 0.2,
    },
    "boosting_dominant": {
        "LightGBM_rank": 0.4,
        "XGBoost_rank": 0.4,
        "ExtraTrees_rank": 0.2,
    },
    "extra_trees_diverse": {
        "LightGBM_rank": 0.4,
        "XGBoost_rank": 0.2,
        "ExtraTrees_rank": 0.4,
    },
    "only_boosting":  {
        "LightGBM_rank": 0.5,
        "XGBoost_rank": 0.5,
    },
    "only_boosting2":  {
            "LightGBM_rank": 0.6,
            "XGBoost_rank": 0.4,
    },
}

In [21]:
ensemble_results = []

for ensemble_name, weights in ensemble_weight_options.items():
    ensemble_auc = (
        oof_scores
        .groupby("validation_step")
        .apply(
            lambda group: roc_auc_score(
                group["y_true"],
                sum(
                    weight * group[column]
                    for column, weight in weights.items()
                ),
            )
        )
    )

    ensemble_results.append({
        "ensemble": ensemble_name,
        "mean_roc_auc": ensemble_auc.mean(),
        "median_roc_auc": ensemble_auc.median(),
        "std_roc_auc": ensemble_auc.std(),
        "worst_roc_auc": ensemble_auc.min(),
        "latest_5_roc_auc": ensemble_auc.tail(5).mean(),
    })

ensemble_results_df = (
    pd.DataFrame(ensemble_results)
    .sort_values(
        ["mean_roc_auc", "latest_5_roc_auc"],
        ascending=False,
    )
)

display(ensemble_results_df)

C:\Users\huawei 14\AppData\Local\Temp\ipykernel_32668\2354125618.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\huawei 14\AppData\Local\Temp\ipykernel_32668\2354125618.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\huawei 14\AppData\Local\Temp\ipykernel_32668\2354125618.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated

,ensemble,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
1,lgbm_dominant,0.995475,0.997577,0.008006,0.968162,0.997855
2,boosting_dominant,0.995395,0.997441,0.008160,0.967572,0.997854
5,only_boosting2,0.995297,0.997112,0.008591,0.965667,0.997840
4,only_boosting,0.995209,0.997090,0.008745,0.965101,0.997831
0,equal,0.995200,0.997398,0.008124,0.967613,0.997597
3,extra_trees_diverse,0.995151,0.997459,0.008023,0.967954,0.997418


In [22]:
standalone_results = []

for model_name in best_model_params:
    model_auc = (
        oof_scores
        .groupby("validation_step")
        .apply(
            lambda group: roc_auc_score(
                group["y_true"],
                group[model_name],
            )
        )
    )

    standalone_results.append({
        "ensemble": model_name,
        "mean_roc_auc": model_auc.mean(),
        "median_roc_auc": model_auc.median(),
        "std_roc_auc": model_auc.std(),
        "worst_roc_auc": model_auc.min(),
        "latest_5_roc_auc": model_auc.tail(5).mean(),
    })

standalone_results_df = pd.DataFrame(standalone_results)

C:\Users\huawei 14\AppData\Local\Temp\ipykernel_32668\3725768638.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\huawei 14\AppData\Local\Temp\ipykernel_32668\3725768638.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
C:\Users\huawei 14\AppData\Local\Temp\ipykernel_32668\3725768638.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated

In [23]:
comparison_results_df = pd.concat(
    [
        standalone_results_df,
        ensemble_results_df,
    ],
    ignore_index=True,
)

comparison_results_df = comparison_results_df.sort_values(
    by=["mean_roc_auc", "latest_5_roc_auc"],
    ascending=False,
)

display(comparison_results_df)

,ensemble,mean_roc_auc,median_roc_auc,std_roc_auc,worst_roc_auc,latest_5_roc_auc
3,lgbm_dominant,0.995475,0.997577,0.008006,0.968162,0.997855
0,LightGBM,0.995398,0.997751,0.008123,0.967539,0.997679
4,boosting_dominant,0.995395,0.997441,0.008160,0.967572,0.997854
5,only_boosting2,0.995297,0.997112,0.008591,0.965667,0.997840
6,only_boosting,0.995209,0.997090,0.008745,0.965101,0.997831
7,equal,0.995200,0.997398,0.008124,0.967613,0.997597
8,extra_trees_diverse,0.995151,0.997459,0.008023,0.967954,0.997418
1,XGBoost,0.994491,0.996804,0.009932,0.960417,0.997647
2,ExtraTrees,0.990361,0.994626,0.010946,0.957531,0.991049
